In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# ── 1. Pull data ──────────────────────────────────────────────────────────────
client = bigquery.Client(project="cbb6790-final-project")
query = """
SELECT *
FROM `cbb6790-final-project.analysis.CBB5790_FinalProject`
"""
df = client.query(query).to_dataframe()

# ── 2. Create binary target ───────────────────────────────────────────────────
df["long_stay"] = (df["icu_length_of_stay"] >= 7).astype(int)
df["gender_bin"] = (df["gender"] == "M").astype(int)

FEATURES = [
    "anchor_age",
    "creatinine_min", "creatinine_max",
    "bun_min", "bun_max",
    "potassium_min", "potassium_max",
    "bicarbonate_min", "bicarbonate_max",
    "sodium_min", "sodium_max",
    "mbp_min", "mbp_mean", "mbp_max",
    "heart_rate_min", "heart_rate_max",
    "urineoutput_24hr",
    "kdigo_stage",
    "gender_bin",
]
TARGET = "long_stay"

# ── 3. Stratified 80/10/10 split within each care unit ───────────────────────
train_dfs, tune_dfs, test_dfs = [], [], []
skipped_units = []

for unit in df["first_careunit"].dropna().unique():
    unit_df = df[df["first_careunit"] == unit].copy()
    if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
        skipped_units.append((unit, len(unit_df), "too small or no outcome variation"))
        continue
    try:
        unit_trainval, unit_test = train_test_split(
            unit_df, test_size=0.10, random_state=42, stratify=unit_df[TARGET]
        )
        unit_train, unit_tune = train_test_split(
            unit_trainval, test_size=0.1111, random_state=42, stratify=unit_trainval[TARGET]
        )
        train_dfs.append(unit_train)
        tune_dfs.append(unit_tune)
        test_dfs.append(unit_test)
    except ValueError as e:
        skipped_units.append((unit, len(unit_df), str(e)))

df_train = pd.concat(train_dfs).reset_index(drop=True)
df_tune  = pd.concat(tune_dfs).reset_index(drop=True)
df_test  = pd.concat(test_dfs).reset_index(drop=True)

print("── Split sizes ──")
print(f"  Train: {len(df_train)} | Tune: {len(df_tune)} | Test: {len(df_test)}")

print("\n── Rows per care unit across splits ──")
for unit in df["first_careunit"].dropna().unique():
    n_train = (df_train["first_careunit"] == unit).sum()
    n_tune  = (df_tune["first_careunit"]  == unit).sum()
    n_test  = (df_test["first_careunit"]  == unit).sum()
    print(f"  {unit}: train={n_train}, tune={n_tune}, test={n_test}")

if skipped_units:
    print("\n── Skipped units ──")
    for unit, n, reason in skipped_units:
        print(f"  {unit} (n={n}): {reason}")

# ── 4. Helper functions ───────────────────────────────────────────────────────
def impute(train_df, eval_df, features):
    """Fit imputer on train, apply to both."""
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(train_df[features])
    X_eval  = imputer.transform(eval_df[features])
    return X_train, X_eval, imputer


def train_local_rf(local_df, features, target, n_estimators=100, max_depth=None):
    """Train a local random forest and return trees + metadata."""
    X, _, imputer = impute(local_df, local_df, features)
    y = local_df[target].values

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X, y)

    return {
        "trees":       model.estimators_,   # list of individual decision trees
        "n_samples":   len(local_df),
        "imputer":     imputer,
        "unit":        None,
        "n_estimators": n_estimators,
    }


def federated_aggregate_rf(client_results):
    """
    Pool all trees from all clients into one forest (weighted by n_samples).
    Each client contributes trees proportional to its dataset size.
    """
    total_n = sum(r["n_samples"] for r in client_results)
    global_trees = []

    for r in client_results:
        # Weight = proportion of total trees to draw from this client
        n_trees_from_client = max(1, round((r["n_samples"] / total_n) * 100))
        # Sample without replacement (or take all if fewer available)
        sampled = r["trees"][:n_trees_from_client]
        global_trees.extend(sampled)

    return global_trees


def predict_with_forest(trees, imputer, eval_df, features):
    """Average predicted probabilities across all trees in the global forest."""
    X = imputer.transform(eval_df[features])
    all_probs = np.array([tree.predict_proba(X)[:, 1] for tree in trees])
    return all_probs.mean(axis=0)


def score_global_rf(trees, global_imputer, eval_df, features, target):
    probs = predict_with_forest(trees, global_imputer, eval_df, features)
    return roc_auc_score(eval_df[target], probs)

# ── 5. Fit global imputer on training data ────────────────────────────────────
global_imputer = SimpleImputer(strategy="median")
global_imputer.fit(df_train[FEATURES])

# ── 6. Tune hyperparameters on validation set ─────────────────────────────────
param_grid = [
    {"n_estimators": 50,  "max_depth": 5},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 100, "max_depth": None},
    {"n_estimators": 200, "max_depth": 10},
]
units = df_train["first_careunit"].dropna().unique()

print("\n── Tuning on validation set ──")
tune_results = {}

for params in param_grid:
    client_results = []
    for unit in units:
        unit_df = df_train[df_train["first_careunit"] == unit].copy()
        if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
            continue
        result = train_local_rf(unit_df, FEATURES, TARGET, **params)
        result["unit"] = unit
        client_results.append(result)

    if not client_results:
        continue

    global_trees = federated_aggregate_rf(client_results)
    tune_auc = score_global_rf(global_trees, global_imputer, df_tune, FEATURES, TARGET)
    key = str(params)
    tune_results[key] = (tune_auc, params)
    print(f"  {params} → Tune AUC: {tune_auc:.4f}")

best_key  = max(tune_results, key=lambda k: tune_results[k][0])
best_params = tune_results[best_key][1]
print(f"\nBest params: {best_params} (Tune AUC: {tune_results[best_key][0]:.4f})")

# ── 7. Final federated training ───────────────────────────────────────────────
print(f"\n── Final federated training ──")
final_client_results = []

for unit in units:
    unit_df = df_train[df_train["first_careunit"] == unit].copy()
    if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
        continue
    result = train_local_rf(unit_df, FEATURES, TARGET, **best_params)
    result["unit"] = unit
    final_client_results.append(result)
    print(f"  ✓ {unit}: n={result['n_samples']}, trees={len(result['trees'])}")

global_trees = federated_aggregate_rf(final_client_results)
print(f"\n  Total trees in global forest: {len(global_trees)}")

# ── 8. AUC Evaluation ─────────────────────────────────────────────────────────
print("\n── Global Model AUC ──")
for split_name, split_df in [("Train", df_train), ("Tune", df_tune), ("Test", df_test)]:
    auc = score_global_rf(global_trees, global_imputer, split_df, FEATURES, TARGET)
    print(f"  {split_name}: {auc:.4f}")

print("\n── Per-Unit Local Model Test AUC ──")
for r in final_client_results:
    try:
        probs = predict_with_forest(r["trees"], r["imputer"], df_test, FEATURES)
        auc   = roc_auc_score(df_test[TARGET], probs)
        print(f"  {r['unit']}: AUC = {auc:.4f}")
    except ValueError:
        print(f"  {r['unit']}: AUC could not be computed")

# ── 9. Feature importance ─────────────────────────────────────────────────────
print("\n── Global Feature Importance (avg across trees) ──")
importance = np.mean([tree.feature_importances_ for tree in global_trees], axis=0)
imp_df = pd.DataFrame({"feature": FEATURES, "importance": importance})
imp_df = imp_df.sort_values("importance", ascending=False)
print(imp_df.to_string(index=False))

── Split sizes ──
  Train: 24456 | Tune: 3061 | Test: 3061

── Rows per care unit across splits ──
  Medical Intensive Care Unit (MICU): train=7614, tune=952, test=952
  Surgical Intensive Care Unit (SICU): train=2509, tune=314, test=314
  Medical/Surgical Intensive Care Unit (MICU/SICU): train=5136, tune=642, test=642
  Trauma SICU (TSICU): train=1864, tune=234, test=234
  Coronary Care Unit (CCU): train=3661, tune=458, test=458
  Cardiac Vascular Intensive Care Unit (CVICU): train=2740, tune=343, test=343
  Neuro Surgical Intensive Care Unit (Neuro SICU): train=310, tune=39, test=39
  Neuro Intermediate: train=416, tune=52, test=52
  Intensive Care Unit (ICU): train=0, tune=0, test=0
  PACU: train=38, tune=5, test=5
  Neuro Stepdown: train=77, tune=10, test=10
  Surgery/Vascular/Intermediate: train=91, tune=12, test=12
  Medicine: train=0, tune=0, test=0
  Surgery/Trauma: train=0, tune=0, test=0
  Medicine/Cardiology Intermediate: train=0, tune=0, test=0

── Skipped units ──
  Intens

In [2]:
# ── 10. Centralized Baseline (Pooled Data) ────────────────────────────────────
print("\n── Centralized Baseline Training ──")

central_imputer = SimpleImputer(strategy="median")
X_train_central = central_imputer.fit_transform(df_train[FEATURES])
y_train_central = df_train[TARGET].values

central_model = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1,
)
central_model.fit(X_train_central, y_train_central)

print("\n── Centralized Model AUC ──")
for split_name, split_df in [("Train", df_train), ("Tune", df_tune), ("Test", df_test)]:
    X_eval = central_imputer.transform(split_df[FEATURES])
    probs = central_model.predict_proba(X_eval)[:, 1]
    auc = roc_auc_score(split_df[TARGET], probs)
    print(f"  {split_name}: {auc:.4f}")

central_importances = central_model.feature_importances_
central_imp_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": central_importances
}).sort_values("importance", ascending=False)

print("\n── Centralized Feature Importance ──")
print(central_imp_df.to_string(index=False))


── Centralized Baseline Training ──

── Centralized Model AUC ──
  Train: 0.9028
  Tune: 0.8069
  Test: 0.8210

── Centralized Feature Importance ──
         feature  importance
     kdigo_stage    0.444864
urineoutput_24hr    0.059412
  heart_rate_max    0.043428
  creatinine_max    0.039962
        mbp_mean    0.038853
      anchor_age    0.038461
         mbp_min    0.038384
  creatinine_min    0.033366
         mbp_max    0.031783
         bun_max    0.029470
  heart_rate_min    0.027772
         bun_min    0.026776
      sodium_max    0.024963
   potassium_max    0.024660
 bicarbonate_min    0.024379
   potassium_min    0.023259
 bicarbonate_max    0.023117
      sodium_min    0.021938
      gender_bin    0.005155
